<a href="https://colab.research.google.com/github/akshaybhatt97/365-DAYS-AI-ML-WITH-AKSHAY/blob/main/Day_106_Cosine_Similarity_Practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 106/365 — Cosine Similarity on Real Text

Yesterday we learned how **token IDs become embedding vectors**.

Today:

# Cosine Similarity

Once text becomes vectors, we need a way to answer:

**How similar are two vectors?**

We will use real SMS messages from the **UCI SMS Spam Collection** and build a tiny similarity-search system.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 1. Load a small real UCI SMS sample

In [2]:
rows = [
    ("ham", "Ok lar... Joking wif u oni..."),
    ("ham", "Nah I don't think he goes to usf, he lives around here though"),
    ("ham", "Even my brother is not like to speak with me."),
    ("ham", "I'm gonna be home soon and i don't want to talk about this stuff anymore tonight, k?"),
    ("ham", "I HAVE A DATE ON SUNDAY WITH WILL!!"),
    ("ham", "Sorry, I'll call later in meeting."),
    ("ham", "Tell where you reached"),
    ("ham", "Fair enough, anything going on?"),
    ("ham", "I'm home."),
    ("ham", "Are you there in room."),
    ("ham", "Can you call me when you reach home"),
    ("ham", "I will be late today. Meeting is still going on"),
    ("ham", "Please call me after your meeting"),
    ("ham", "I reached home safely"),
    ("spam", "Free entry in 2 a wkly comp to win FA Cup final tkts. Text FA to 87121."),
    ("spam", "WINNER!! As a valued network customer you have been selected to receive a prize reward!"),
    ("spam", "URGENT! You have won a FREE membership in our Prize Jackpot! Txt CLAIM now."),
    ("spam", "Your free ringtone is waiting to be collected. Text MIX to verify."),
    ("spam", "Congratulations! You have won a cash prize. Reply CLAIM now."),
    ("spam", "Free offer! Text WIN now to claim your reward."),
    ("spam", "Claim your free prize now. Text WIN to receive your reward."),
]

df = pd.DataFrame(
    rows,
    columns=["label","text"]
)

df.head()


,label,text
0,ham,Ok lar... Joking wif u oni...
1,ham,"Nah I don't think he goes to usf, he lives aro..."
2,ham,Even my brother is not like to speak with me.
3,ham,I'm gonna be home soon and i don't want to tal...
4,ham,I HAVE A DATE ON SUNDAY WITH WILL!!


## 2. Cosine similarity formula

For vectors `A` and `B`:

`cos(A,B) = (A · B) / (||A|| ||B||)`

Interpretation:

- close to `1` → very similar direction
- close to `0` → weak similarity
- close to `-1` → opposite direction

For many text-retrieval systems, larger cosine similarity means more similar content.


In [3]:
A = np.array([1.0, 2.0, 3.0])
B = np.array([1.2, 2.1, 2.8])

dot_product = np.dot(A, B)

cosine = (
    dot_product
    / (
        np.linalg.norm(A)
        * np.linalg.norm(B)
    )
)

print(
    "Cosine similarity:",
    round(cosine,4)
)


Cosine similarity: 0.9968


## 3. Convert real messages into vectors

For this lesson we use **TF-IDF vectors**.

TF-IDF is not a modern neural embedding model, but it gives us real document vectors that make cosine similarity easy to inspect.

The cosine-similarity calculation is the same core operation we can later apply to dense neural embeddings.


In [4]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1,2),
    stop_words="english"
)

message_vectors = (
    vectorizer.fit_transform(
        df["text"]
    )
)

print(
    "Vector matrix shape:",
    message_vectors.shape
)


Vector matrix shape: (21, 148)


## 4. Compare two real messages

In [5]:
message_1 = (
    "Please call me after your meeting"
)

message_2 = (
    "Can you call me when you reach home"
)

vector_1 = vectorizer.transform(
    [message_1]
)

vector_2 = vectorizer.transform(
    [message_2]
)

score = cosine_similarity(
    vector_1,
    vector_2
)[0,0]

print(
    "Cosine similarity:",
    round(score,4)
)


Cosine similarity: 0.0


## 5. Build similarity search

Query:

`please call me after the meeting`

Steps:

1. Convert the query to a vector.
2. Compare it with all stored message vectors.
3. Calculate cosine similarity.
4. Sort by score.
5. Return the most similar messages.


In [6]:
query = (
    "please call me after the meeting"
)

query_vector = (
    vectorizer.transform(
        [query]
    )
)

scores = cosine_similarity(
    query_vector,
    message_vectors
).ravel()

top_indices = (
    np.argsort(scores)[::-1][:5]
)

results = df.iloc[
    top_indices
].copy()

results["similarity"] = (
    scores[top_indices]
)

results[
    ["label","text","similarity"]
]


,label,text,similarity
12,ham,Please call me after your meeting,1.000000
11,ham,I will be late today. Meeting is still going on,0.314423
5,ham,"Sorry, I'll call later in meeting.",0.309059
19,spam,Free offer! Text WIN now to claim your reward.,0.000000
20,spam,Claim your free prize now. Text WIN to receive...,0.000000


This gives us the basic retrieval pipeline:

`Query`

→ Vector

→ Similarity Search

→ Ranked Results


## 6. Try a spam-style query

In [7]:
query = (
    "claim your free prize now"
)

query_vector = vectorizer.transform(
    [query]
)

scores = cosine_similarity(
    query_vector,
    message_vectors
).ravel()

top_indices = (
    np.argsort(scores)[::-1][:5]
)

results = df.iloc[
    top_indices
].copy()

results["similarity"] = (
    scores[top_indices]
)

results[
    ["label","text","similarity"]
]


,label,text,similarity
20,spam,Claim your free prize now. Text WIN to receive...,0.597391
16,spam,URGENT! You have won a FREE membership in our ...,0.222683
18,spam,Congratulations! You have won a cash prize. Re...,0.181120
19,spam,Free offer! Text WIN now to claim your reward.,0.180390
15,spam,WINNER!! As a valued network customer you have...,0.076088


## 7. Why cosine instead of just magnitude?

Consider:

`A = [1,2]`

`B = [10,20]`

The vectors have very different lengths.

But they point in the same direction.

Cosine similarity is therefore `1.0`.


In [8]:
A = np.array([[1,2]])
B = np.array([[10,20]])

print(
    cosine_similarity(
        A,
        B
    )[0,0]
)


0.9999999999999999


## 8. Connection to neural embeddings

A modern embedding model may produce vectors such as:

`embedding("machine learning")`

and:

`embedding("artificial intelligence")`

Those vectors may have hundreds or thousands of dimensions.

We can still calculate cosine similarity between them.

The embedding model changes.

The similarity formula stays the same.


## 9. Connection to RAG

A simplified RAG retrieval pipeline is:

`Documents`

→ split into chunks

→ generate embeddings

→ store vectors

Then:

`User Query`

→ query embedding

→ cosine similarity / nearest-neighbor search

→ retrieve relevant chunks

→ send retrieved context to the LLM

This is where embeddings start becoming an actual AI system component.


## 10. Connection to vector databases

At small scale, we can compare a query against every stored vector.

At large scale — millions or billions of vectors — that becomes expensive.

Vector databases use specialized indexes to perform nearest-neighbor search efficiently.

Popular similarity choices include:

- cosine similarity
- dot product
- Euclidean distance


## Simple takeaway

Embeddings give us vectors.

Cosine similarity tells us how similarly those vectors point.

The practical pipeline is:

`Text → Vector → Cosine Similarity → Ranked Results`

> **This simple operation is one of the foundations of semantic search, vector databases and RAG.**


## References

- UCI SMS Spam Collection:
  https://archive.ics.uci.edu/dataset/228/sms+spam+collection

- scikit-learn cosine similarity:
  https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html

- TF-IDF:
  https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

- Sentence-BERT:
  https://arxiv.org/abs/1908.10084
